# The bee's knees: reading MRI with an anatomical mirror

![An artistic knee joint with mirrored MRI planes](https://raw.githubusercontent.com/cubres/rsna-knee-abnormality-detection/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/figures/knee-anatomical-mirror-hero-v1.png)

*AI-generated artistic illustration, not measured MRI. [Prompt and artwork provenance](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/ARTWORK_PROVENANCE.json).*

A knee MRI is a collection of acquisitions, not one picture. This notebook shows how a strong public method turns that collection into a finding vector: choose the slices, preserve their physical geometry, read adjacent triplets, pool the evidence, and average a plain and an anatomically mirrored view. The implementation is deliberately readable enough to fork.

The checkpoint and mirror recipe come from [nartaa / Danial Zakaria's scored V2](https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror?scriptVersionId=356175397), building on dreaddevelopment's Raptor and timm's CoAtNet. **The upstream notebook displayed 0.949. That is nartaa's score, not ours.**

## Results so far

| Entry | Score | Basis |
|---|---|---|
| nartaa V2: upstream recipe and checkpoint | 0.949 | Their score on their notebook. Attributed here, not reproduced |
| This public notebook (card) | 0.941 | Official rows 56759401 (V31), 56779708 and 56779710 (V32), all 0.941 |
| Account's verified best | 0.943 | Private notebook, row 56859861 (V8) |
| Public frontier, observed 2026-10-08 | Six notebooks at 0.950 (blends) | Public leaderboard. Not ours |
| Private V14 | pending | Row 56948803; no score |
| This version, V34 | pending | No score is claimed until a completed official submission exists |

Earlier versions and their displayed results are recorded in the preserved history at the end. Version 34 changes the active code path. Its own native GPU run and official submission are still to come.

## What changed in version 34

Version 34 starts its clock in the first statement of the notebook, before any import, staging or CUDA work, so the whole time budget includes setup. Inference runs as the leader of its own process group under the same owned-process watchdog our private campaign uses. The watchdog cleans up the whole group on success as well as on failure. The output is staged and checked for coverage, schema, row order, finiteness, source and checkpoint identity, and file digest. Only then is `/kaggle/working/submission.csv` written, with exclusive creation.

Version 34 also replaces the strict reader with a tolerant one. Section 3 explains why that matters.

[Scientific contract](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/SCIENTIFIC_CONTRACT.md) · [Licenses and model-use terms](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSE_SCOPE.md) · The active reader source is the cell in section 3. Its sha256 is pinned in the launcher.


## Read the experiment before running it

![MRI volume, windows, two views, attention and ranks](https://raw.githubusercontent.com/cubres/rsna-knee-abnormality-detection/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/figures/anatomical_mirror_pipeline_v2.png)

| Choice | This baseline | Why it matters |
|---|---|---|
| Checkpoint | One selected public SWA checkpoint, exact V1 hash | Reproducible identity; this is not a new independent model |
| Physical field of view | 140 mm center crop | Pixels alone do not fix the anatomical field of view |
| Source volume | 96 slices at 384 × 384 | Preserve the pretrained geometry and five acquisition slots |
| Model input | 94 adjacent triplets, center crop to 320 × 320 | Keep neighboring-slice information without resizing again |
| Mirror | Acquisition channels for sagittal centers; image width otherwise | A reflection must follow the acquisition plane |
| Pooling | Finding-specific attention over windows | Each finding can attend to different evidence |
| Combine views | Mean sigmoid probabilities, then per-finding ordinal ranks | This is not logit averaging or clinical calibration |

There is no training step and no model download. Attach [the publisher's original weights dataset V1](https://www.kaggle.com/datasets/nartaa/rsna-knee-publication-swa-weights-20261007/versions/1), use competition inputs you are authorized to access, and choose one or more **T4 GPUs**. The qualified environment used two. The qualified container has Torch `2.11.0+cu128` and timm `1.0.29`; library checks fail before any model is loaded.

The baseline runs all current test rows in `inference` mode. Set `MODE="canary"` for a three-study smoke test. That writes a separately named file and is not an accuracy measurement. A notebook run only creates predictions. It does not submit to the competition.


## 1. A small configuration you can actually use

The first code cell starts the clock. Its first statement reads the monotonic time before any import, staging or CUDA work, so the budget covers setup as well as inference. A competition rerun (`KAGGLE_IS_COMPETITION_RERUN` set to true or 1) uses the published 32,400-second GPU limit. A commit uses the 1,800-second notebook limit. Each budget keeps a 50-second margin, so the whole-run deadlines are 32,350 s and 1,750 s. An unknown flag value stops the run instead of guessing.

Change `MODE` and `PREP_WORKERS` for a bounded smoke test or a preparation-speed experiment. Change paths only if your attached data layout differs. The baseline geometry and equal view weights live in the reader in section 3. If you change them, update `READER_SHA256` and document the change before treating a fork as reproducible.


In [ ]:
_T0 = __import__("time").monotonic()  # the owned-run clock: the first statement of this notebook
import datetime, hashlib, json, os, sys, time
from pathlib import Path

CONFIG = {
    "MODE": "inference",                 # "inference": all test rows and the root submission; "canary": three-study smoke test
    "PREP_WORKERS": 2,                   # the qualified value; supported 1, 2, 4
    "MAX_CANARY_STUDIES": 3,             # canary mode only
    "INPUT_ROOT": Path("/kaggle/input"),
    "COMPETITION_ROOT": None,            # None: detect the mounted competition
    "OUTPUT_ROOT": Path("/kaggle/working"),
    "NOTEBOOK_LIMIT_SECONDS": 1800,      # the commit runtime limit
    "COMPETITION_LIMIT_SECONDS": 32400,  # the published nine-hour GPU limit for competition reruns
    "SAFETY_MARGIN_SECONDS": 50,
    "EXPECTED_TORCH": "2.11.0+cu128",
    "EXPECTED_TIMM": "1.0.29",
}
READER_SHA256 = "e9ec648e1b2f6121f36bbe6ae806a5714571db83804c719ea02d5c4bf7fe47d7"
OWNED_WATCHDOG_SHA256 = "54636411f9f20aeacdeec784b3cb4b5e5a3fbb8c6ea2ec334f0d0d41b6e6130e"
CHECKPOINT_SHA256 = "7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef"
LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


require(CONFIG["MODE"] in ("canary", "inference"), "Unknown mode")
require(CONFIG["PREP_WORKERS"] in (1, 2, 4), "Supported preparation workers: 1, 2, 4")
require(1 <= CONFIG["MAX_CANARY_STUDIES"] <= 16, "Canary size must be 1..16")
_RERUN_RAW = os.environ.get("KAGGLE_IS_COMPETITION_RERUN")
_RERUN_NORMALIZED = None if _RERUN_RAW is None else _RERUN_RAW.lower()
require(_RERUN_NORMALIZED in (None, "", "false", "0", "true", "1"),
        "Unknown competition-rerun flag; no budget fallback")
COMPETITION_RERUN = _RERUN_NORMALIZED in ("true", "1")
SELECTED_LIMIT = CONFIG["COMPETITION_LIMIT_SECONDS"] if COMPETITION_RERUN else CONFIG["NOTEBOOK_LIMIT_SECONDS"]
WHOLE_BUDGET = SELECTED_LIMIT - CONFIG["SAFETY_MARGIN_SECONDS"]
FULL_DEADLINE = _T0 + WHOLE_BUDGET
RUN_TAG = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
SOURCE_DIR = CONFIG["OUTPUT_ROOT"] / "public_v34_source"
SOURCE_DIR.mkdir(parents=True, exist_ok=False)
READER_PATH = SOURCE_DIR / "public_mirror_reader.py"
WATCHDOG_PATH = SOURCE_DIR / "owned_watchdog.py"
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_HUB_DISABLE_TELEMETRY="1")
print({"budget_seconds": WHOLE_BUDGET, "competition_rerun": COMPETITION_RERUN, "mode": CONFIG["MODE"],
       "competition_submission": False, "official_score": None})


## 2. Respect the acquisition geometry

| Slot | Source slice indices | Count |
|---|---|---:|
| Sagittal, fluid-sensitive | 0–25 | 26 |
| Sagittal, non-fluid-sensitive | 26–47 | 22 |
| Coronal, fluid-sensitive | 48–65 | 18 |
| Coronal, non-fluid-sensitive | 66–77 | 12 |
| Axial, any sequence | 78–95 | 18 |

We prefer the first suitable unused series in metadata order, falling back within the requested plane. DICOM position and orientation order the slices, with `InstanceNumber` as a fallback. Apply the modality LUT and MONOCHROME1 inversion before robust per-series intensity scaling. Sample the 2–98% index span, use the selected images' 2nd/98th pixel percentiles, crop a 140 mm field of view, then area-resize and quantize to uint8.

The volume layout survives missing acquisition slots. A study with no usable acquisition fails instead of receiving model priors. Window selection preserves the upstream span and can cross slot boundaries; the center determines the mirror operation.

![Acquisition-channel reversal and width reflection before normalization](https://raw.githubusercontent.com/cubres/rsna-knee-abnormality-detection/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/figures/mirror_before_normalization_v2.png)

Mirror the **uint8** triplet first. Then crop the central 320 × 320 region (32 pixels from each edge of 384 × 384), and only then apply the three channel-specific ImageNet means and standard deviations. Reversing channels after normalization would alter the recipe because those statistics differ.


## 3. The reader, and why tolerance matters

The next cell writes the exact version-34 reader (sha256 `e9ec648e…`). The launcher verifies those bytes before it starts. The same tolerant reader, its diff from the strict version, the offline test harness and the visible-run receipts are published in the [official-results folder](https://github.com/cubres/rsna-knee-abnormality-detection/tree/b8e1d6e89c391e27821a41455d5f42336f70420d/official-results-2026-10-08) of the [repository](https://github.com/cubres/rsna-knee-abnormality-detection). The architecture and the happy-path arithmetic are the same as before. What changed is how the reader treats bad input.

An earlier private version of this pipeline ran the same architecture under a strict reader. That reader stopped the whole job on an unreadable DICOM header, an unreadable or non-2D pixel array, a study with no usable slice, any per-study model error, or a device count other than two. Its visible three-study run passed. Its rerun on the hidden test cohort, about 1,300 studies, ended with an unhandled error and no score. We cannot see the hidden log, so this is our most likely explanation rather than a proven one. A cohort that large is very likely to contain at least one file that a strict reader rejects, and one such file should not zero the submission.

The public reference handles these cases deliberately. An unreadable header keeps its file as a neutral record. An unreadable slice is skipped and its slot stays zero. A study whose volume cannot be built is predicted from a zero volume and mask. A failed model call gives that study 0.5 probabilities. Every study still receives a prediction. In offline tests with synthetic DICOM, this reader matched the reference on truncated, garbage, multi-frame, NaN and missing-series studies. Its output was identical to the strict reader's on healthy studies. Those tests do not measure accuracy, and the GPU path still needs its own native run.

The general lesson is that a competition reader is part of the scoring pipeline. Its failure policy should be stated, tested on bad inputs, and chosen so that every study stays in the output. The functions to read are `build_volume` (the fault policy), `prepare_windows` and `predict_pair` (the model path), and `run` (coverage and ranks).


In [ ]:
%%writefile /kaggle/working/public_v34_source/public_mirror_reader.py
"""Original inference implementation of the qualified public Knee reader.

Architecture and geometry: dreaddevelopment's Raptor CoAtNet/MIL lineage.
Selected learned weights and plain/mirror recipe: nartaa public V2,
https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror?scriptVersionId=356175397
Notebook source is Apache-2.0. Weights have separate research/educational terms,
including this competition. This module grants no rights to OAI source data.

Fault policy (V14): the same tolerant semantics as the public reference's
reader (its header record, per-slice pixel skip and whole-study fallback).
A header failure keeps the file as a (0.0, path, 0.5) record; a pixel failure
leaves that slot zero and advances the slot index; a study whose volume cannot
be built is predicted from a zero volume and zero mask; a model failure gives
that study 0.5 probabilities. No study is dropped, and any CUDA device count
of at least one is accepted.

Importing this file performs no model load, network request or native inference.
Only main() starts an explicitly requested local Kaggle experiment.
"""
from __future__ import annotations

import argparse
from concurrent.futures import ThreadPoolExecutor
import glob
import hashlib
import json
import os
from pathlib import Path
import queue
import threading
import time

PUBLIC_REF = "nartaa/rsna-knee-0949-anatomical-mirror"
PUBLIC_VERSION = 2
PUBLIC_SESSION_ID = 356175397
PUBLIC_NOTEBOOK_SHA256 = "8f1f96530b08543bd6caeed48489eea7c01657d6994f22e21a8400047fd4e100"
PUBLIC_INFERENCE_CELL_SHA256 = "bcd864cbf349882e876959b49cb348cbb89b4c62fc204d4e4261049c5e3a79c5"
DATASET_REF = "nartaa/rsna-knee-publication-swa-weights-20261007"
DATASET_VERSION = 1
CHECKPOINT_NAME = "raptor_ft_alldata_t16_blendjev_oai_d96_r384_swa.pt"
CHECKPOINT_SHA256 = "7e5315dad125b99fc65b340b3de41de628e9be51ff5835355dd61c86472244ef"
ARCH = "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k"
LABELS = ("ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
          "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture")
SLOTS = (("Sagittal", 1, 26), ("Sagittal", 0, 22), ("Coronal", 1, 18),
         ("Coronal", 0, 12), ("Axial", -1, 18))
CORPUS_RES = 384
INPUT_RES = 320
SLICE_COUNT = 96
WINDOW_COUNT = 94
CROP_MM = 140.0
IMAGE_MEAN = (0.485, 0.456, 0.406)
IMAGE_STD = (0.229, 0.224, 0.225)
FAULT_POLICY = ("V14 reference-tolerant: header failure keeps (0.0, 0.5) record; pixel failure skips slice "
                "(zero slot, index advances); study build failure gives zero volume and mask; model failure "
                "gives 0.5 probabilities; no study is dropped")
PROBABILITY_DEFAULT = 0.5


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def mirror_axis_for_center(center):
    require(type(center) is int and 0 <= center < SLICE_COUNT, "Invalid centre slice")
    return "channels" if center < 48 else "width"


def find_attached_checkpoint(root):
    candidates = (
        root / DATASET_REF.split("/")[1] / CHECKPOINT_NAME,
        root / "datasets" / DATASET_REF / CHECKPOINT_NAME,
    )
    available = [path for path in candidates if path.is_file()]
    require(bool(available), "Attach the qualified public dataset V1; checkpoint is absent")
    path = available[0]
    with path.open("rb") as stream:
        digest = hashlib.file_digest(stream, "sha256").hexdigest()
    require(digest == CHECKPOINT_SHA256, "Checkpoint bytes differ from the qualified public V1")
    return path


def dependencies():
    import cv2
    import numpy as np
    import pandas as pd
    import pydicom
    try:
        from pydicom.pixels import apply_modality_lut
    except ImportError:
        from pydicom.pixel_data_handlers.util import apply_modality_lut
    import timm
    import torch
    return cv2, np, pd, pydicom, apply_modality_lut, timm, torch


def make_model(checkpoint, device, timm, torch):
    nn = torch.nn

    class FindingAttentionReader(nn.Module):
        def __init__(self, backbone):
            super().__init__()
            self.backbone = backbone
            self.norm = nn.LayerNorm(backbone.num_features)
            self.att = nn.Sequential(nn.Linear(backbone.num_features, 256), nn.Tanh(),
                                     nn.Dropout(0.2), nn.Linear(256, 12))
            self.clsW = nn.Parameter(torch.zeros(12, backbone.num_features))
            self.clsb = nn.Parameter(torch.zeros(12))
            self.n = 12

        def forward(self, windows):
            batch, count = windows.shape[:2]
            flat = windows.flatten(0, 1)
            encoded = torch.cat([self.backbone(chunk) for chunk in flat.split(16)], dim=0)
            features = self.norm(encoded.view(batch, count, -1))
            attention = torch.softmax(self.att(features), dim=1)
            pooled = torch.einsum("bkn,bkf->bnf", attention, features)
            return (pooled * self.clsW).sum(-1) + self.clsb

    # Tensor-only deserialization is intentionally stricter than the public source.
    # Its compatibility with this checkpoint must pass the native canary first.
    state = torch.load(checkpoint, map_location="cpu", weights_only=True)
    require(isinstance(state, dict) and "model" in state, "Unsupported checkpoint container")
    require(state.get("arch", ARCH) == ARCH, "Checkpoint architecture differs")
    require(int(state.get("res", CORPUS_RES)) == CORPUS_RES, "Checkpoint training resolution differs")
    backbone = timm.create_model(ARCH, pretrained=False, num_classes=0, in_chans=3,
                                 img_size=INPUT_RES, global_pool="avg")
    model = FindingAttentionReader(backbone)
    model.load_state_dict(state["model"], strict=True)
    del state
    model.eval().to(device)
    require(all(not module.training for module in model.modules()), "Model has a training module")
    return model


def _new_counters():
    return {"missing_slots": 0, "selected_slices": 0, "header_failures": 0, "pixel_failures": 0,
            "study_build_failed": False, "study_build_error_type": None, "study_build_error": None}


def _zero_study(np, counters):
    return (np.zeros((SLICE_COUNT, CORPUS_RES, CORPUS_RES), dtype=np.uint8),
            np.zeros(SLICE_COUNT, dtype=np.uint8), counters)


def build_volume(study_id, rows, series_root, deps):
    """Return (volume, mask, counters). Never raises (reference fault policy).

    Per-header and per-pixel failures are absorbed exactly where the reference
    absorbs them. Any exception that still escapes marks the whole study as
    failed: the volume and mask are zero, as in the reference's study fallback.
    """
    np = deps[1]
    try:
        return _build_volume_checked(study_id, rows, series_root, deps)
    except Exception as error:
        counters = _new_counters()
        counters.update(study_build_failed=True, study_build_error_type=type(error).__name__,
                        study_build_error=str(error)[:300])
        return _zero_study(np, counters)


def _build_volume_checked(study_id, rows, series_root, deps):
    cv2, np, _, pydicom, modality_lut, _, _ = deps
    volume = np.zeros((SLICE_COUNT, CORPUS_RES, CORPUS_RES), dtype=np.uint8)
    used = set()
    offset = 0
    counters = _new_counters()

    def header(path):
        # Reference _hdr: an unreadable header keeps the file as (0.0, path, 0.5, False).
        try:
            data = pydicom.dcmread(path, stop_before_pixels=True)
            orientation = getattr(data, "ImageOrientationPatient", None)
            position = getattr(data, "ImagePositionPatient", None)
            if orientation is not None and position is not None and len(orientation) == 6:
                normal = np.cross(np.asarray(orientation[:3], dtype=float),
                                  np.asarray(orientation[3:], dtype=float))
                coordinate = float(np.dot(np.asarray(position, dtype=float), normal))
            else:
                coordinate = float(getattr(data, "InstanceNumber", 0) or 0)
            spacing = getattr(data, "PixelSpacing", None)
            spacing = float(spacing[0]) if spacing is not None else 0.5
            return coordinate, path, spacing, True
        except Exception:
            return 0.0, path, 0.5, False

    def pixels(item):
        # Reference _px: an unreadable pixel array gives None, and that slice is skipped.
        _, path, spacing, _ = item
        try:
            data = pydicom.dcmread(path)
            image = modality_lut(data.pixel_array, data).astype(np.float32)
            if str(getattr(data, "PhotometricInterpretation", "")) == "MONOCHROME1":
                image = image.max() - image
            return image, spacing
        except Exception:
            return None, spacing

    for plane, fluid, count in SLOTS:
        candidates = [row for row in rows if row["Anatomical_Plane"] == plane
                      and str(row["SeriesInstanceUID"]) not in used]
        preferred = [row for row in candidates
                     if fluid in (0, 1) and int(row.get("Fluid_Sensitive", 0) or 0) == fluid]
        chosen = (preferred or candidates)[0] if candidates else None
        if chosen is None:
            counters["missing_slots"] += 1
            offset += count
            continue
        series_id = str(chosen["SeriesInstanceUID"])
        used.add(series_id)
        # Reference uses glob.glob on the series directory; the same listing order is kept.
        paths = glob.glob(f"{series_root}/{study_id}/{series_id}/*.dcm")
        if not paths:
            counters["missing_slots"] += 1
            offset += count
            continue
        with ThreadPoolExecutor(max_workers=16) as executor:
            records = list(executor.map(header, paths))
        counters["header_failures"] += sum(1 for record in records if not record[3])
        ok_spacings = [record[2] for record in records if record[3]]
        median_spacing = float(np.median(ok_spacings)) if ok_spacings else 0.5
        records.sort(key=lambda record: record[0])
        total = len(records)
        low = int(total * 0.02)
        high = max(int(total * 0.98) - 1, low)
        indices = np.linspace(low, high, count).round().astype(int) if total > 1 else [0] * count
        selected = [records[min(int(index), total - 1)] for index in indices]
        with ThreadPoolExecutor(max_workers=8) as executor:
            arrays = list(executor.map(pixels, selected))
        counters["pixel_failures"] += sum(1 for image, _ in arrays if image is None)
        valid_images = [image for image, _ in arrays if image is not None]
        if valid_images:
            low_value, high_value = np.percentile(np.concatenate([image.ravel() for image in valid_images]),
                                                [2.0, 98.0])
        else:
            low_value, high_value = 0.0, 1.0
        for image, spacing in arrays:
            if image is None:
                offset += 1
                continue
            normalized = np.clip((image - low_value) / (high_value - low_value + 1e-6), 0, 1)
            # A non-2D array fails the crop in the reference (unpacking h, w), so the whole study falls back.
            require(normalized.ndim == 2, "Non-2D DICOM pixel array")
            spacing = spacing if spacing > 0 else median_spacing
            size = min(int(round(CROP_MM / max(spacing, 1e-3))), *normalized.shape)
            require(size > 0, "Empty physical crop")
            top, left = ((normalized.shape[0] - size) // 2, (normalized.shape[1] - size) // 2)
            cropped = normalized[top:top + size, left:left + size]
            resized = cv2.resize(cropped, (CORPUS_RES, CORPUS_RES), interpolation=cv2.INTER_AREA)
            volume[offset] = (resized * 255).astype(np.uint8)
            offset += 1
            counters["selected_slices"] += 1
    require(offset == SLICE_COUNT, "Study slot count differs")
    mask = (volume.reshape(SLICE_COUNT, -1).sum(1) > 0).astype(np.uint8)
    return volume, mask, counters


def prepare_windows(volume, mask, device, mirrored, np, torch):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, len(volume)))
    low, high = int(valid.min()), int(valid.max())
    centers = list(range(low + 1, high))
    if not centers:
        centers = [max(1, min((low + high) // 2, SLICE_COUNT - 2))]
    indices = np.linspace(0, len(centers) - 1, WINDOW_COUNT).round().astype(int)
    centers = np.asarray([centers[int(index)] for index in indices], dtype=np.int64)
    triplets = torch.from_numpy(np.ascontiguousarray(
        volume[np.stack([centers - 1, centers, centers + 1], axis=1)])).to(device)
    require(tuple(triplets.shape) == (WINDOW_COUNT, 3, CORPUS_RES, CORPUS_RES), "Triplet shape differs")
    if mirrored:
        sagittal = torch.from_numpy(np.asarray([
            mirror_axis_for_center(int(center)) == "channels" for center in centers
        ], dtype=bool)).to(device).view(-1, 1, 1, 1)
        # Reflect uint8 acquisition channels first; ImageNet channel statistics differ.
        triplets = torch.where(sagittal, triplets.flip(1), triplets.flip(-1))
    border = (CORPUS_RES - INPUT_RES) // 2
    triplets = triplets[:, :, border:border + INPUT_RES, border:border + INPUT_RES]
    # CPU-generated LUT reproduces the public float32 normalization order exactly.
    values = torch.from_numpy(np.arange(256, dtype=np.uint8).astype(np.float32) / 255.0)
    table = torch.stack([values, values, values], dim=0).view(3, 256, 1)
    means = torch.tensor(IMAGE_MEAN, dtype=torch.float32).view(3, 1, 1)
    scales = torch.tensor(IMAGE_STD, dtype=torch.float32).view(3, 1, 1)
    table = ((table - means) / scales).view(3, 256).contiguous().to(device)
    windows = torch.stack([table[channel][triplets[:, channel].long()] for channel in range(3)], dim=1)
    require(tuple(windows.shape) == (WINDOW_COUNT, 3, INPUT_RES, INPUT_RES), "Model input shape differs")
    return windows


def predict_pair(model, volume, mask, device, np, torch):
    require(all(not module.training for module in model.modules()), "TTA changed model mode")
    predictions = []
    with torch.no_grad():
        for mirrored in (False, True):
            windows = prepare_windows(volume, mask, device, mirrored, np, torch)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(windows.unsqueeze(0)).float()
                require(bool(torch.isfinite(logits).all()), "Nonfinite model logits")
                predictions.append(torch.sigmoid(logits)[0].cpu().numpy())
            del windows
    values = 0.5 * (predictions[0] + predictions[1])
    require(values.shape == (12,) and np.isfinite(values).all(), "Invalid study probabilities")
    return values, predictions[0], predictions[1]


def _device_name(torch, index):
    try:
        return torch.cuda.get_device_name(index)
    except Exception:
        return "unavailable"


def run(args):
    started = time.monotonic()
    os.environ.setdefault("HF_HUB_OFFLINE", "1")
    os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
    os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
    deps = dependencies()
    _, np, pd, _, _, timm, torch = deps
    require(torch.cuda.is_available(), "This experiment requires CUDA/fp16")
    gpu_count = int(torch.cuda.device_count())
    require(gpu_count >= 1, "At least one CUDA device is required")
    device_names = [_device_name(torch, index) for index in range(gpu_count)]
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    checkpoint = find_attached_checkpoint(args.input_root)
    root = args.competition_root
    series_root = root / "test_series"
    if not series_root.is_dir():
        series_root = root / "test_images"
    require(series_root.is_dir(), "Test DICOM directory is absent")
    test = pd.read_csv(root / "test.csv", dtype={"StudyInstanceUID": str})
    sample = pd.read_csv(root / "sample_submission.csv", dtype={"StudyInstanceUID": str})
    all_ids = test["StudyInstanceUID"].tolist()
    expected_columns = ["StudyInstanceUID", *LABELS]
    require(len(set(all_ids)) == len(all_ids), "Duplicate test study ID")
    require(sample.columns.tolist() == expected_columns, "Sample submission label/schema drift")
    require(sample["StudyInstanceUID"].tolist() == all_ids, "Test/sample row order differs")
    ids = all_ids if args.mode == "inference" else all_ids[:args.max_studies]
    require(bool(ids), "No studies selected")
    series = pd.read_csv(root / "test_series.csv", dtype={"StudyInstanceUID": str, "SeriesInstanceUID": str})
    grouped = {key: frame.to_dict("records") for key, frame in series.groupby("StudyInstanceUID", sort=False)}
    # A study without series rows is predicted from a zero volume, as in the reference (no drop).
    studies_without_series = sum(1 for uid in ids if uid not in grouped)
    models = [make_model(checkpoint, f"cuda:{index}", timm, torch) for index in range(gpu_count)]
    jobs = queue.Queue(maxsize=4)
    results, model_fallbacks, consumer_errors = {}, [], []
    lock = threading.Lock()

    def consumer(index):
        while True:
            item = jobs.get()
            try:
                if item is None:
                    return
                study_id, volume, mask, counters = item
                try:
                    values, plain, mirrored = predict_pair(models[index], volume, mask,
                                                           f"cuda:{index}", np, torch)
                    fallback = None
                except Exception as error:
                    # Reference arm default: a study whose model call fails keeps 0.5 probabilities.
                    values = plain = mirrored = np.full(12, PROBABILITY_DEFAULT, dtype=np.float32)
                    fallback = {"study_id": study_id, "device": f"cuda:{index}",
                                "error_type": type(error).__name__, "message": str(error)[:300]}
                    try:
                        torch.cuda.empty_cache()
                    except Exception:
                        pass
                with lock:
                    results[study_id] = (values, plain, mirrored, counters)
                    if fallback is not None:
                        model_fallbacks.append(fallback)
                    count = len(results)
                    if count % 25 == 0 or count == len(ids):
                        print(json.dumps({"completed": count, "total": len(ids),
                                          "elapsed_s": round(time.monotonic() - started, 2),
                                          "model_fallbacks": len(model_fallbacks)}), flush=True)
            except Exception as error:
                # Keep the consumer alive so the producer is never stranded on a full queue.
                with lock:
                    consumer_errors.append({"study_id": item[0] if item else None,
                                            "error_type": type(error).__name__, "message": str(error)[:300]})
            finally:
                jobs.task_done()

    threads = [threading.Thread(target=consumer, args=(index,)) for index in range(gpu_count)]
    for thread in threads:
        thread.start()
    try:
        with ThreadPoolExecutor(max_workers=args.prep_workers) as executor:
            # At most eight queued preparation futures; no fork after Torch/CUDA initialization.
            for offset in range(0, len(ids), 8):
                batch = ids[offset:offset + 8]
                futures = [executor.submit(build_volume, uid, grouped.get(uid, []), series_root, deps) for uid in batch]
                for uid, future in zip(batch, futures):
                    try:
                        volume, mask, counters = future.result()
                    except Exception as error:
                        volume, mask, counters = _zero_study(np, _new_counters())
                        counters.update(study_build_failed=True, study_build_error_type=type(error).__name__,
                                        study_build_error=str(error)[:300])
                    jobs.put((uid, volume, mask, counters))
    finally:
        for _ in threads:
            jobs.put(None)
        jobs.join()
        for thread in threads:
            thread.join()
    # Coverage guard: a study without a result gets the reference default instead of being dropped.
    missing_results = [uid for uid in ids if uid not in results]
    for uid in missing_results:
        default = np.full(12, PROBABILITY_DEFAULT, dtype=np.float32)
        results[uid] = (default, default, default, _new_counters())
        model_fallbacks.append({"study_id": uid, "device": None, "error_type": "MissingResult",
                                "message": "no consumer result; reference default applied"})
    require(set(results) == set(ids) and len(results) == len(ids), "Incomplete study coverage")
    probabilities = np.stack([results[uid][0] for uid in ids])
    require(probabilities.shape == (len(ids), 12) and np.isfinite(probabilities).all(), "Output shape/finite gate")
    require(bool(((probabilities >= 0) & (probabilities <= 1)).all()), "Probability range gate")
    # Exact public ordinal rank transform; do not quietly substitute a different tie policy.
    ranks = probabilities.argsort(0).argsort(0).astype(np.float64) / max(1, len(ids) - 1)
    # Reference guard: any non-finite rank becomes the neutral 0.5.
    ranks[~np.isfinite(ranks)] = 0.5
    output = pd.DataFrame(ranks.astype(np.float32), columns=LABELS)
    output.insert(0, "StudyInstanceUID", ids)
    require(output.columns.tolist() == expected_columns and output["StudyInstanceUID"].tolist() == ids,
            "Final row/schema gate")
    args.output_dir.mkdir(parents=True, exist_ok=True)
    name = "submission.csv" if args.mode == "inference" else "canary_predictions.csv"
    destination = args.output_dir / name
    require(not destination.exists(), "Output already exists; preserve it and use a fresh directory")
    with destination.open("x") as stream:
        output.to_csv(stream, index=False)
    raw_path = args.output_dir / "probabilities.npz"
    with raw_path.open("xb") as stream:
        np.savez_compressed(stream, study_uids=np.asarray(ids, dtype=str), labels=np.asarray(LABELS),
                            values=probabilities, plain=np.stack([results[uid][1] for uid in ids]),
                            mirrored=np.stack([results[uid][2] for uid in ids]))
    receipt = {"mode": args.mode, "full_test_coverage": ids == all_ids, "studies": len(ids),
               "native_inference_completed": True, "competition_submitted": False,
               "official_score": None, "public_reference_score": 0.949,
               "public_ref": PUBLIC_REF, "public_version": PUBLIC_VERSION,
               "public_numeric_session_id": PUBLIC_SESSION_ID, "checkpoint_sha256": CHECKPOINT_SHA256,
               "dataset_ref": DATASET_REF, "dataset_version": DATASET_VERSION,
               "source_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),
               "versions": {"torch": torch.__version__, "timm": timm.__version__, "numpy": np.__version__},
               "gpu_count": gpu_count, "devices": device_names,
               "elapsed_s": time.monotonic() - started, "prep_workers": args.prep_workers,
               "fault_policy": FAULT_POLICY,
               "missing_slots": sum(results[uid][3]["missing_slots"] for uid in ids),
               "selected_slices": sum(results[uid][3]["selected_slices"] for uid in ids),
               "header_failures": sum(results[uid][3]["header_failures"] for uid in ids),
               "pixel_failures": sum(results[uid][3]["pixel_failures"] for uid in ids),
               "study_build_failures": sum(1 for uid in ids if results[uid][3]["study_build_failed"]),
               "studies_without_series_metadata": studies_without_series,
               "model_fallback_studies": len(model_fallbacks),
               "model_fallback_examples": model_fallbacks[:5],
               "consumer_error_count": len(consumer_errors), "consumer_error_examples": consumer_errors[:5],
               "prediction_file_sha256": hashlib.sha256(destination.read_bytes()).hexdigest()}
    receipt_path = args.output_dir / "native_inference_receipt.json"
    with receipt_path.open("x") as stream:
        json.dump(receipt, stream, indent=2, sort_keys=True)
        stream.write("\n")
    print(json.dumps(receipt, sort_keys=True), flush=True)


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--input-root", type=Path, default=Path("/kaggle/input"))
    parser.add_argument("--competition-root", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    parser.add_argument("--mode", choices=("canary", "inference"), default="canary")
    parser.add_argument("--max-studies", type=int, default=3)
    parser.add_argument("--prep-workers", type=int, choices=(1, 2, 4), default=2)
    args = parser.parse_args()
    require(1 <= args.max_studies <= 16, "Canary study count must be 1..16")
    run(args)


if __name__ == "__main__":
    main()


## 4. The owned process group

A timeout on one child is not enough. A reader that starts workers can leave descendants behind. A notebook that exports a file before those processes are cleaned up can hide a problem. The next cell writes the owned-process watchdog (sha256 `54636411…`), the same module our private campaign uses. It runs the reader as the leader of a fresh process group, enforces an absolute deadline, and on success terminates the group and verifies that teardown has resolved before the launcher accepts any output. It is Apache-2.0, like the reader.


In [ ]:
%%writefile /kaggle/working/public_v34_source/owned_watchdog.py
# SPDX-License-Identifier: Apache-2.0
"""Bound one owned POSIX process group; never signal a process by its name.

The supervisor holds its group-leader PID until acknowledged. The parent never
reaps it before error/timeout signalling, so that PID cannot be reused while the
ownership lease is held. Descendants that change their process group or session
are outside this contract. By default, success attests only the direct command's
completion. Optional success cleanup terminates its still-owned group before
returning success; it cannot certify absence of escaped descendants. This helper
performs no filesystem cleanup. Deadlines are checked throughout, but POSIX process
creation and other system calls are not themselves interruptibly bounded.
"""
from __future__ import annotations

from dataclasses import dataclass
import json
import math
import os
import select
import signal
import subprocess
import sys
import time


class OwnershipLostError(RuntimeError):
    """Ownership could not be proved; no subsequent group signal is allowed."""


class OwnedProcessError(RuntimeError):
    def __init__(self, message, receipt):
        super().__init__(message)
        self.receipt = receipt


class OwnedProcessTimeout(OwnedProcessError):
    pass


@dataclass(frozen=True)
class DiagnosticBudget:
    remaining_seconds: int
    child_wall_seconds: int
    absolute_monotonic_deadline: float


def diagnostic_budget(native_started_monotonic, *, now=None):
    """Inclusive 850-second parent budget; retain five seconds over the child."""
    current = time.monotonic() if now is None else now
    for value in (native_started_monotonic, current):
        if isinstance(value, bool) or not isinstance(value, (int, float)) or not math.isfinite(value):
            raise ValueError("diagnostic clock values must be finite numbers")
    elapsed = current - native_started_monotonic
    if elapsed < 0:
        raise ValueError("diagnostic clock moved before its start")
    remaining = math.floor(850 - elapsed)
    if remaining < 65:
        raise ValueError("bootstrap left fewer than 65 seconds for child and watchdog")
    child = remaining - 5
    if not 60 <= child <= 850:
        raise ValueError("child budget violates the native parser contract")
    return DiagnosticBudget(remaining, child, native_started_monotonic + 850)


_SUPERVISOR = r'''
import json, os, signal, subprocess, sys
# A caught TERM handler keeps this ownership anchor alive until group KILL.
# POSIX exec resets caught handlers to default in the actual payload process.
signal.signal(signal.SIGTERM, lambda _number, _frame: None)
status_fd, acknowledgement_fd = int(sys.argv[1]), int(sys.argv[2])
command, payload_fds = json.loads(sys.argv[3]), tuple(json.loads(sys.argv[4]))
def report(value):
    raw = (json.dumps(value, separators=(",", ":")) + "\n").encode()
    if len(raw) > 4096:
        raise RuntimeError("supervisor metadata exceeds pipe bound")
    os.write(status_fd, raw)
try:
    child = subprocess.Popen(command, pass_fds=payload_fds)
    report({"stage": "started", "command_pid": child.pid})
    result = child.wait()
    report({"stage": "finished", "returncode": result})
except BaseException as exc:
    report({"stage": "supervisor_error", "error_type": type(exc).__name__})
# Hold the unreaped group-leader identity through the parent's decision.
acknowledgement = os.read(acknowledgement_fd, 1)
sys.exit(0 if acknowledgement == b"S" else 125)
'''


class _Lease:
    def __init__(self, process):
        self.process = process
        self.pid = process.pid
        self.signals = []
        self.verify()

    def verify(self):
        if self.process.returncode is not None:
            raise OwnershipLostError("owned leader was already reaped")
        try:
            sid, pgid = os.getsid(self.pid), os.getpgid(self.pid)
        except OSError as exc:
            raise OwnershipLostError("owned leader identity unavailable") from exc
        if sid != self.pid or pgid != self.pid:
            raise OwnershipLostError("owned leader session/group identity changed")
        return {"session_id": sid, "process_group_id": pgid}

    def send_group(self, number):
        proof = self.verify()
        event = {"signal": signal.Signals(number).name,
                 "target_process_group_id": self.pid,
                 "observed_session_id": proof["session_id"],
                 "observed_process_group_id": proof["process_group_id"]}
        try:
            os.killpg(self.pid, number)
        except ProcessLookupError:
            self.signals.append({**event, "group_present": False})
        else:
            self.signals.append({**event, "group_present": True})


def _terminate_owned(lease, deadline):
    """Keep the leader unreaped until both bounded group signals are attempted."""
    record = {"ownership_proved": False, "signals": [], "leader_reaped": False,
              "unresolved": False}
    try:
        lease.verify()
        record["ownership_proved"] = True
        lease.send_group(signal.SIGTERM)
        grace = min(0.25, max(0.0, deadline - time.monotonic()) / 4)
        if grace:
            time.sleep(grace)
        # The still-unreaped leader reserves the group ID even after SIGTERM.
        lease.send_group(signal.SIGKILL)
        record["signals"] = list(lease.signals)
        remaining = max(0.0, deadline - time.monotonic())
        lease.process.wait(timeout=remaining)
        record["leader_reaped"] = True
    except (OwnershipLostError, subprocess.TimeoutExpired, OSError, ChildProcessError) as exc:
        record["signals"] = list(lease.signals)
        record["unresolved"] = True
        record["error_type"] = type(exc).__name__
        # Ownership loss is a hard stop, never a reason for a fallback signal.
    return record


def run_owned(command, *, timeout, absolute_deadline=None, cleanup_on_success=False,
              cwd=None, env=None, stdout=None, stderr=None, passthrough_fds=()):
    """Run a command in a fresh owned session, preserving an inclusive deadline.

Return a bounded metadata receipt on success. Nonzero exit, timeout or an
observer error stops only the freshly owned group and raises with its receipt.
With cleanup_on_success=True, keep the successful command's supervisor
unacknowledged and unreaped through TERM and KILL, and require resolved teardown
before declaring direct-command success. The default retains the original
acknowledgement-only behavior. No group signal is issued after acknowledgement
or reaping, and an unresolved teardown is never retried.
SIGCHLD must retain its default disposition; callers must not reap this private
supervisor from another thread or install a reaping handler during this call.
"""
    if os.name != "posix" or not hasattr(os, "killpg"):
        raise RuntimeError("owned process-group watchdog requires POSIX")
    if signal.getsignal(signal.SIGCHLD) != signal.SIG_DFL:
        raise OwnershipLostError("default SIGCHLD disposition is required")
    if type(cleanup_on_success) is not bool:
        raise ValueError("cleanup_on_success must be a boolean")
    if isinstance(timeout, bool) or not isinstance(timeout, (int, float)) or not math.isfinite(timeout) or timeout <= 0.25:
        raise ValueError("watchdog timeout must be finite and greater than 0.25 seconds")
    if not isinstance(command, (list, tuple)) or not command or any(not isinstance(part, str) or not part for part in command):
        raise ValueError("command must be a nonempty argument vector")
    fds = tuple(passthrough_fds)
    if any(isinstance(fd, bool) or not isinstance(fd, int) or fd < 3 for fd in fds) or len(set(fds)) != len(fds):
        raise ValueError("passthrough file descriptors must be unique integers at least three")
    started = time.monotonic()
    deadline = started + timeout
    if absolute_deadline is not None:
        if isinstance(absolute_deadline, bool) or not isinstance(absolute_deadline, (int, float)) or not math.isfinite(absolute_deadline):
            raise ValueError("absolute watchdog deadline must be a finite monotonic clock value")
        deadline = min(deadline, absolute_deadline)
    effective_timeout = deadline - started
    if effective_timeout <= 0.25:
        raise ValueError("absolute deadline left insufficient watchdog time before spawn")
    reserve = min(2.0, max(0.2, effective_timeout / 5))
    observation_deadline = deadline - reserve
    status_read, status_write = os.pipe()
    acknowledgement_read, acknowledgement_write = os.pipe()
    process = lease = None
    receipt = {"status": "UNRESOLVED", "ownership_strategy": "unreaped-owned-supervisor",
               "command_returncode": None, "timed_out": False, "group_signals": [],
               "effective_timeout_seconds": effective_timeout,
               "absolute_monotonic_deadline": deadline,
               "cleanup_on_success": cleanup_on_success}
    supervisor_acknowledged = False
    teardown_attempted = False
    closed = set()
    def close_fd(fd):
        if fd not in closed:
            os.close(fd)
            closed.add(fd)
    try:
        process = subprocess.Popen(
            [sys.executable, "-u", "-c", _SUPERVISOR, str(status_write),
             str(acknowledgement_read), json.dumps(list(command)), json.dumps(fds)],
            cwd=cwd, env=env, stdout=stdout, stderr=stderr,
            start_new_session=True, pass_fds=(status_write, acknowledgement_read, *fds),
        )
        close_fd(status_write)
        close_fd(acknowledgement_read)
        lease = _Lease(process)
        receipt.update({"owned_leader_pid": lease.pid, "owned_session_id": lease.pid,
                        "owned_process_group_id": lease.pid})
        os.set_blocking(status_read, False)
        buffered = b""
        command_started = False
        while True:
            left = observation_deadline - time.monotonic()
            if left <= 0:
                receipt["timed_out"] = True
                raise TimeoutError("owned command exceeded its observation deadline")
            ready, _, _ = select.select([status_read], [], [], min(left, 0.05))
            if not ready:
                continue
            raw = os.read(status_read, 4096)
            if not raw:
                raise RuntimeError("owned supervisor metadata pipe closed before completion")
            buffered += raw
            if len(buffered) > 8192:
                raise RuntimeError("owned supervisor metadata exceeded its bound")
            while b"\n" in buffered:
                line, buffered = buffered.split(b"\n", 1)
                message = json.loads(line)
                if message.get("stage") == "started":
                    if command_started or set(message) != {"stage", "command_pid"} or type(message["command_pid"]) is not int:
                        raise RuntimeError("invalid owned-command start metadata")
                    command_started = True
                    receipt["command_pid"] = message["command_pid"]
                elif message.get("stage") == "finished":
                    if not command_started or set(message) != {"stage", "returncode"} or type(message["returncode"]) is not int:
                        raise RuntimeError("invalid owned-command completion metadata")
                    receipt["command_returncode"] = message["returncode"]
                    if message["returncode"] != 0:
                        raise RuntimeError("owned command returned a nonzero status")
                    lease.verify()
                    if cleanup_on_success:
                        # The zero-exit payload is reaped by the supervisor, while
                        # this fresh session leader still anchors group ownership.
                        teardown_attempted = True
                        teardown = _terminate_owned(lease, deadline)
                        receipt["teardown"] = teardown
                        receipt["group_signals"] = teardown["signals"]
                        if (teardown["unresolved"] or not teardown["ownership_proved"]
                                or not teardown["leader_reaped"]):
                            if teardown.get("error_type") == "TimeoutExpired":
                                raise TimeoutError("owned success cleanup exceeded its deadline")
                            raise RuntimeError("owned success cleanup did not resolve group teardown")
                        completed = time.monotonic()
                        if completed > deadline:
                            raise TimeoutError("owned success cleanup completed after its inclusive deadline")
                        receipt.update({"status": "PASS_OWNED_COMMAND", "leader_reaped": True,
                                        "success_completion_strategy": "direct-zero-exit-owned-group-teardown",
                                        "supervisor_acknowledged": False,
                                        "supervisor_returncode": process.returncode,
                                        "elapsed_seconds": completed - started})
                        return receipt
                    os.write(acknowledgement_write, b"S")
                    supervisor_acknowledged = True
                    process.wait(timeout=max(0.0, deadline - time.monotonic()))
                    if process.returncode != 0:
                        raise RuntimeError("owned supervisor did not acknowledge success")
                    completed = time.monotonic()
                    if completed > deadline:
                        raise TimeoutError("owned command acknowledgement completed after its inclusive deadline")
                    receipt.update({"status": "PASS_OWNED_COMMAND", "leader_reaped": True,
                                    "elapsed_seconds": completed - started})
                    return receipt
                else:
                    raise RuntimeError("owned supervisor reported an error or unknown stage")
    except BaseException as exc:
        if isinstance(exc, (TimeoutError, subprocess.TimeoutExpired)):
            receipt["timed_out"] = True
        # An acknowledged supervisor may already be reaped; never signal it.
        if lease is not None and not supervisor_acknowledged and not teardown_attempted:
            teardown_attempted = True
            teardown = _terminate_owned(lease, deadline)
            receipt["teardown"] = teardown
            receipt["group_signals"] = teardown["signals"]
        elif process is not None and "teardown" not in receipt:
            receipt["teardown"] = {"unresolved": True, "reason": "ownership unavailable or success already acknowledged"}
        receipt.update({"status": "TIMEOUT_OWNED_COMMAND" if receipt["timed_out"] else "FAIL_OWNED_COMMAND",
                        "error_type": type(exc).__name__, "elapsed_seconds": time.monotonic() - started})
        failure = OwnedProcessTimeout if receipt["timed_out"] else OwnedProcessError
        raise failure("owned watchdog stopped without declaring native success", receipt) from exc
    finally:
        for fd in (status_read, status_write, acknowledgement_read, acknowledgement_write):
            close_fd(fd)


## 4. Let each finding choose its windows

CoAtNet encodes the triplets in chunks of 16. LayerNorm and a small attention head give each finding its own softmax over the windows. The weighted sum of window features produces one vector per finding; a finding-specific classifier turns it into a logit. Dropout is disabled in evaluation mode.

```text
attention[k, c] = softmax over k (attention_head(feature[k]))
pooled[c]       = sum over k (attention[k, c] * feature[k])
logit[c]        = classifier[c](pooled[c])
probability[c]  = (sigmoid(plain_logit[c]) + sigmoid(mirror_logit[c])) / 2
```

The final CSV uses the upstream per-column `argsort().argsort() / max(1, N-1)` transform. These ordinal ranks are useful for ranking; they are not calibrated probabilities. Preserve the source's NumPy tie behavior. Canary ranks depend on the canary's N and therefore cannot establish full-test prediction equivalence.


## 5. Run in stages and export last

The launcher checks the pinned runtime, the reader and watchdog bytes, and the competition mount before any checkpoint is loaded. It starts the reader under the owned watchdog with the absolute deadline set in the first cell. The reader writes to a fresh stage directory, `/kaggle/working/public_v34_stage_<run>/`, and never to the submission path. After the child's process group is cleaned up and its receipt is read, the staged file is validated: sample columns and row order, finite values in [0, 1], full coverage, and the SHA-256 bound in the native receipt. Only then is `/kaggle/working/submission.csv` written, with exclusive creation. If any check fails, no submission file appears.

The checkpoint requires the publisher's weights dataset attachment. Source SHA-256 and the checkpoint digest are distinct pins. The reader hashes the actual checkpoint file before deserialization. No code license grants rights to redistribute checkpoint bytes or controlled-access records.


In [ ]:
import importlib.metadata
import importlib.util
import subprocess

import numpy as np
import pandas as pd

# Pins are checked before any child process is started. Distribution metadata only: no CUDA or model work here.
require(importlib.metadata.version("torch") == CONFIG["EXPECTED_TORCH"] and
        importlib.metadata.version("timm") == CONFIG["EXPECTED_TIMM"],
        "Use the qualified Torch 2.11.0+cu128 / timm 1.0.29 container")


def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


require(sha256_file(READER_PATH) == READER_SHA256, "Staged reader bytes differ from the qualified V14 reader")
require(sha256_file(WATCHDOG_PATH) == OWNED_WATCHDOG_SHA256, "Staged watchdog bytes differ from the qualified owned watchdog")
COMPETITION_ROOT = None
for _root in ([CONFIG["INPUT_ROOT"] / "competitions/rsna-knee-abnormality-detection",
               CONFIG["INPUT_ROOT"] / "rsna-knee-abnormality-detection"]
              if CONFIG["COMPETITION_ROOT"] is None else [Path(CONFIG["COMPETITION_ROOT"])]):
    if all((_root / name).is_file() for name in ("test.csv", "sample_submission.csv", "test_series.csv")):
        COMPETITION_ROOT = _root
        break
require(COMPETITION_ROOT is not None, "Attach the authorized competition inputs or configure the actual mount")
require(not COMPETITION_RERUN or CONFIG["MODE"] == "inference", "Competition reruns require full inference")

STAGE_DIR = CONFIG["OUTPUT_ROOT"] / ("public_v34_stage_" + RUN_TAG)
OUTPUT_DIR = STAGE_DIR / CONFIG["MODE"]
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
PREDICTION_NAME = "submission.csv" if CONFIG["MODE"] == "inference" else "canary_predictions.csv"
ROOT_SUBMISSION = CONFIG["OUTPUT_ROOT"] / "submission.csv"
require(not ROOT_SUBMISSION.exists() and not ROOT_SUBMISSION.is_symlink(),
        "A root submission already exists; preserve it and start a fresh run")


def save_new(path, value):
    with Path(path).open("x") as stream:
        json.dump(value, stream, sort_keys=True, indent=2, allow_nan=False)
        stream.write("\n")


_remaining = FULL_DEADLINE - time.monotonic()
require(_remaining >= 65, "Insufficient owned budget left for the child and the watchdog")
_name = "_owned_public_v34_watchdog"
require(_name not in sys.modules, "Owned watchdog identity occupied")
_spec = importlib.util.spec_from_file_location(_name, WATCHDOG_PATH)
_watchdog = importlib.util.module_from_spec(_spec)
sys.modules[_name] = _watchdog
_spec.loader.exec_module(_watchdog)
require(Path(_watchdog.__file__).resolve() == WATCHDOG_PATH.resolve(), "Owned watchdog origin changed")

command = [sys.executable, "-u", str(READER_PATH), "--input-root", str(CONFIG["INPUT_ROOT"]),
           "--competition-root", str(COMPETITION_ROOT), "--output-dir", str(OUTPUT_DIR),
           "--mode", CONFIG["MODE"], "--max-studies", str(CONFIG["MAX_CANARY_STUDIES"]),
           "--prep-workers", str(CONFIG["PREP_WORKERS"])]
child_env = dict(os.environ, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_HUB_DISABLE_TELEMETRY="1")
try:
    with (OUTPUT_DIR / "reader_worker.log").open("x") as log:
        # The owned process group is the whole child tree; cleanup is resolved before any file is accepted.
        owned = _watchdog.run_owned(command, timeout=_remaining + 2.0, absolute_deadline=FULL_DEADLINE + 2.0,
                                    cleanup_on_success=True, cwd=str(SOURCE_DIR), env=child_env,
                                    stdout=log, stderr=subprocess.STDOUT)
    require(owned.get("status") == "PASS_OWNED_COMMAND" and owned.get("command_returncode") == 0 and
            owned.get("cleanup_on_success") is True and owned.get("leader_reaped") is True and
            owned.get("teardown", {}).get("unresolved") is False,
            "Owned inference lifecycle incomplete; no submission is exported")
    require(time.monotonic() <= FULL_DEADLINE, "Whole budget exceeded before validation")
except BaseException as error:
    if not (STAGE_DIR / "public_v34_failure.json").exists():
        save_new(STAGE_DIR / "public_v34_failure.json",
                 dict(status="FAIL_PUBLIC_V34_INFERENCE", error_type=type(error).__name__,
                      message=str(error)[:500], elapsed_s=time.monotonic() - _T0,
                      competition_submitted=False, official_score=None))
    raise

# Validation of the staged file. Nothing is written to the working root until every check passes.
receipt = json.loads((OUTPUT_DIR / "native_inference_receipt.json").read_bytes())
require(receipt.get("native_inference_completed") is True and receipt.get("mode") == CONFIG["MODE"] and
        receipt.get("source_sha256") == READER_SHA256 and receipt.get("checkpoint_sha256") == CHECKPOINT_SHA256 and
        receipt.get("competition_submitted") is False and receipt.get("official_score") is None,
        "Native receipt drift")
sample = pd.read_csv(COMPETITION_ROOT / "sample_submission.csv", dtype={"StudyInstanceUID": str})
test_ids = pd.read_csv(COMPETITION_ROOT / "test.csv", dtype={"StudyInstanceUID": str})["StudyInstanceUID"].tolist()
expected_columns = ["StudyInstanceUID", *LABELS]
require(sample.columns.tolist() == expected_columns and sample["StudyInstanceUID"].tolist() == test_ids,
        "Sample schema or row order differs from test.csv")
expected_rows = len(test_ids) if CONFIG["MODE"] == "inference" else min(len(test_ids), CONFIG["MAX_CANARY_STUDIES"])
staged = OUTPUT_DIR / PREDICTION_NAME
prediction = pd.read_csv(staged, dtype={"StudyInstanceUID": str})
require(prediction.columns.tolist() == expected_columns and len(prediction) == expected_rows and
        prediction["StudyInstanceUID"].tolist() == test_ids[:expected_rows], "Prediction schema, coverage or order drift")
if CONFIG["MODE"] == "inference":
    require(receipt.get("full_test_coverage") is True and receipt.get("studies") == len(test_ids),
            "Incomplete full-test coverage")
values = prediction[LABELS].to_numpy(dtype=np.float64)
require(values.shape == (expected_rows, 12) and np.isfinite(values).all() and
        ((values >= 0) & (values <= 1)).all(), "Nonfinite or out-of-range values")
staged_bytes = staged.read_bytes()
require(hashlib.sha256(staged_bytes).hexdigest() == receipt["prediction_file_sha256"],
        "Prediction bytes differ from the native receipt")

# The last write of the notebook: the exclusive root export, only in full inference mode.
if CONFIG["MODE"] == "inference":
    with ROOT_SUBMISSION.open("xb") as export:
        export.write(staged_bytes)
    require(hashlib.sha256(ROOT_SUBMISSION.read_bytes()).hexdigest() == receipt["prediction_file_sha256"],
            "Root export differs from the validated staged file")
print({"status": "public V34 inference validated", "mode": CONFIG["MODE"], "rows": expected_rows,
       "findings": 12, "schema_and_order": "PASS", "finite_and_range": "PASS",
       "root_submission_exported": CONFIG["MODE"] == "inference", "competition_submitted": False,
       "official_score": None})


## 6. Verify the file without displaying study identifiers

A successful run is not sufficient by itself. The launcher has already checked the sample columns and row order, finite values in [0, 1], full coverage in inference mode, and the file digest bound into the native receipt. The cell below re-reads the exported file and prints only aggregate counts and hashes.

The staged directory contains `probabilities.npz`, the reader's log and the receipt. Some of these carry study identifiers. Use them only in your authorized workspace. Do not publish them as demonstration data or as raw notebook output. A canary file must never be submitted as a full-test prediction.


In [ ]:
# Aggregate verification only: no study identifiers or prediction values are printed.
if CONFIG["MODE"] == "inference":
    _root_bytes = ROOT_SUBMISSION.read_bytes()
    _root_receipt = json.loads((OUTPUT_DIR / "native_inference_receipt.json").read_bytes())
    require(hashlib.sha256(_root_bytes).hexdigest() == _root_receipt["prediction_file_sha256"], "Root file drift")
    print({"root_submission_sha256": hashlib.sha256(_root_bytes).hexdigest(), "bytes": len(_root_bytes),
           "studies": _root_receipt["studies"], "full_test_coverage": _root_receipt["full_test_coverage"],
           "model_fallback_studies": _root_receipt.get("model_fallback_studies"),
           "study_build_failures": _root_receipt.get("study_build_failures"),
           "header_failures": _root_receipt.get("header_failures"),
           "pixel_failures": _root_receipt.get("pixel_failures"),
           "gpu_count": _root_receipt.get("gpu_count"), "competition_submitted": False, "official_score": None})
else:
    print({"canary": True, "root_submission_exported": False, "official_score": None})


## What to change in a useful fork

Begin with this exact baseline and save its receipt. For a new experiment, change one mechanism, write down the hypothesis before running, and compare against the same baseline on a suitable independent validation split.

| Question | Small, interpretable experiment | Main caution |
|---|---|---|
| Does the mirrored view contribute independent information? | Compare plain, mirror and their equal probability mean | Keep the checkpoint, studies and preprocessing identical |
| Is acquisition geometry responsible for the gain? | Compare the documented plane-aware mirror with an explicitly defined alternative | Channel statistics and reflection order must stay explicit |
| Do two trained readers add diversity? | Test a preregistered blend with an independently trained, permitted checkpoint | Shared training data or ancestry can reduce diversity |
| Can preparation use fewer CPU threads? | Change `PREP_WORKERS` and measure elapsed time/peak memory | A source check does not measure speed |

Do not tune weights repeatedly on a reused exposed panel or infer causality from a single public leaderboard change. The publisher's Gold58 set was reused for selection, so it is not independent cross-validation. The older plain-versus-affine experiment belongs to a separate training question; it is not evidence that this mirror recipe improved or failed.


## Credits, changes and required acknowledgement

## Upstream lineage

- **nartaa / Danial Zakaria**: selected learned checkpoint and plain/anatomical-mirror inference recipe in [RSNA Knee 0949 Anatomical Mirror, V2](https://www.kaggle.com/code/nartaa/rsna-knee-0949-anatomical-mirror?scriptVersionId=356175397), Apache-2.0. The displayed 0.949 score belongs to that upstream notebook.
- **dreaddevelopment**: [Raptor CoAtNet/MIL recipe and public baseline weights](https://www.kaggle.com/datasets/dreaddevelopment/raptor-knee-widedense). The selected nartaa weights are the publisher's retrained checkpoints, not these original baseline weights.
- **timm / Ross Wightman and contributors**: [CoAtNet implementation and pretrained component lineage](https://github.com/huggingface/pytorch-image-models). Third-party components retain their own licenses.
- The checkpoint publisher credits weak-label contributions from [stevenleehans](https://www.kaggle.com/datasets/stevenleehans/rsna-knee-llm-report-labels), [pilkwang](https://www.kaggle.com/datasets/pilkwang/rsna-knee-llm-labels), and [riadmohamed42's JEV labels and verification](https://www.kaggle.com/code/riadmohamed42/jev-knee-labels-verification), together with the publisher's report-derived labels. These tables and source reports are not included.
- **RSNA and competition data contributors**: data use remains subject to the [RSNA Knee competition rules](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/rules).

## Original implementation and changes

`public_mirror_reader.py` (version 34, sha256 `e9ec648e…`) is an original readable implementation of the visible upstream recipe, not a copy of private campaign code or an independently trained model. Relative to the strict reader of earlier versions, it adopts the public reference's fault semantics: unreadable headers keep a neutral record, unreadable slices are skipped, unbuildable studies are predicted from zero inputs, failed model calls give 0.5, and no study is dropped. It adds checkpoint and source identity constants, schema and coverage gates, tensor-only deserialization, bounded thread queues without multiprocessing, separate canary output names and create-only outputs. It does not execute the upstream fast-reader payload. The owned-process watchdog (`owned_watchdog.py`, sha256 `54636411…`) is original Apache-2.0 code. The selected checkpoint bytes, geometry and aggregation formulas remain attributed to the public references. These differences require native validation; this version makes no reproduced-score or bit-equivalence claim.

## Publisher's required OAI / NDA acknowledgement

The following paragraph is preserved verbatim from the upstream publisher's credits and dataset description. It acknowledges the upstream research lineage; distributing this reader does not assert that this repository obtained or redistributed controlled-access OAI records.

Data and/or research tools used in the preparation of this manuscript were obtained and analyzed from the controlled access datasets distributed from the Osteoarthritis Initiative (OAI), a data repository housed within the NIMH Data Archive (NDA). OAI is a collaborative informatics system created by the National Institute of Mental Health and the National Institute of Arthritis, Musculoskeletal and Skin Diseases (NIAMS) to provide a worldwide resource to quicken the pace of biomarker identification, scientific investigation and OA drug development. Dataset identifier(s): 10.15154/0hcg-f676.

[Shared NDA Study 3407](https://nda.nih.gov/study.html?id=3407) · [DOI: 10.15154/0hcg-f676](https://doi.org/10.15154/0hcg-f676). Official acknowledgement source: [NDA manuscript preparation](https://nda.nih.gov/nda/manuscript-preparation).

The upstream weights package grants only its stated model-use permission. OAI access and redistribution remain governed by NDA. This notebook contains no OAI or competition MRI, reports, source label tables, participant identifiers or learned checkpoint payloads.


### Source and model licenses stay separate

- Active original reader and upstream source lineage: **Apache-2.0**. [Complete Apache license](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSES/Apache-2.0.txt).
- Newly written notebook explanations/configuration/diagrams: **MIT**, copyright 2026 cubres. [MIT text](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSES/MIT.txt).
- Selected checkpoint: publisher's **Other / research and educational model-use terms**, linked as original V1; no checkpoint redistribution or sublicensing grant is assumed.
- Historical inactive source below retains all original notices, including its DINOv3 license. This refresh does not relicense that archive.

The publisher's model-use statement is:

> These learned weights are provided for research and educational use, including the RSNA Knee competition. This model-use permission grants no rights to obtain, publish, redistribute or re-identify OAI participant data, source MRI images, reports or other controlled-access records. Access to OAI data remains governed by NDA's terms. Third-party source code and pretrained components retain their respective licences. No CC0 or other open-data licence is applied to OAI data or raw corpora by this model release.

[Versioned model-use explanation](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/MODEL_USAGE.md) · [License boundaries](https://github.com/cubres/rsna-knee-abnormality-detection/blob/454a541e769edec700ea25d2e649c1e9c43d4e71/public-mirror-reader-2026-10-08/LICENSE_SCOPE.md) · The version-34 reader is the cell in section 3. No weights, source MRI, reports, label tables or participant records are embedded in this notebook source.


<details><summary>Complete Apache-2.0 license for the active reader</summary>

```text
                                 Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editorial revisions, annotations, elaborations, or other modifications
      represent, as a whole, an original work of authorship. For the purposes
      of this License, Derivative Works shall not include works that remain
      separable from, or merely link (or bind by name) to the interfaces of,
      the Work and Derivative Works thereof.

      "Contribution" shall mean any work of authorship, including
      the original version of the Work and any modifications or additions
      to that Work or Derivative Works thereof, that is intentionally
      submitted to Licensor for inclusion in the Work by the copyright owner
      or by an individual or Legal Entity authorized to submit on behalf of
      the copyright owner. For the purposes of this definition, "submitted"
      means any form of electronic, verbal, or written communication sent
      to the Licensor or its representatives, including but not limited to
      communication on electronic mailing lists, source code control systems,
      and issue tracking systems that are managed by, or on behalf of, the
      Licensor for the purpose of discussing and improving the Work, but
      excluding communication that is conspicuously marked or otherwise
      designated in writing by the copyright owner as "Not a Contribution."

      "Contributor" shall mean Licensor and any individual or Legal Entity
      on behalf of whom a Contribution has been received by Licensor and
      subsequently incorporated within the Work.

   2. Grant of Copyright License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      copyright license to reproduce, prepare Derivative Works of,
      publicly display, publicly perform, sublicense, and distribute the
      Work and such Derivative Works in Source or Object form.

   3. Grant of Patent License. Subject to the terms and conditions of
      this License, each Contributor hereby grants to You a perpetual,
      worldwide, non-exclusive, no-charge, royalty-free, irrevocable
      (except as stated in this section) patent license to make, have made,
      use, offer to sell, sell, import, and otherwise transfer the Work,
      where such license applies only to those patent claims licensable
      by such Contributor that are necessarily infringed by their
      Contribution(s) alone or by combination of their Contribution(s)
      with the Work to which such Contribution(s) was submitted. If You
      institute patent litigation against any entity (including a
      cross-claim or counterclaim in a lawsuit) alleging that the Work
      or a Contribution incorporated within the Work constitutes direct
      or contributory patent infringement, then any patent licenses
      granted to You under this License for that Work shall terminate
      as of the date such litigation is filed.

   4. Redistribution. You may reproduce and distribute copies of the
      Work or Derivative Works thereof in any medium, with or without
      modifications, and in Source or Object form, provided that You
      meet the following conditions:

      (a) You must give any other recipients of the Work or
          Derivative Works a copy of this License; and

      (b) You must cause any modified files to carry prominent notices
          stating that You changed the files; and

      (c) You must retain, in the Source form of any Derivative Works
          that You distribute, all copyright, patent, trademark, and
          attribution notices from the Source form of the Work,
          excluding those notices that do not pertain to any part of
          the Derivative Works; and

      (d) If the Work includes a "NOTICE" text file as part of its
          distribution, then any Derivative Works that You distribute must
          include a readable copy of the attribution notices contained
          within such NOTICE file, excluding those notices that do not
          pertain to any part of the Derivative Works, in at least one
          of the following places: within a NOTICE text file distributed
          as part of the Derivative Works; within the Source form or
          documentation, if provided along with the Derivative Works; or,
          within a display generated by the Derivative Works, if and
          wherever such third-party notices normally appear. The contents
          of the NOTICE file are for informational purposes only and
          do not modify the License. You may add Your own attribution
          notices within Derivative Works that You distribute, alongside
          or as an addendum to the NOTICE text from the Work, provided
          that such additional attribution notices cannot be construed
          as modifying the License.

      You may add Your own copyright statement to Your modifications and
      may provide additional or different license terms and conditions
      for use, reproduction, or distribution of Your modifications, or
      for any such Derivative Works as a whole, provided Your use,
      reproduction, and distribution of the Work otherwise complies with
      the conditions stated in this License.

   5. Submission of Contributions. Unless You explicitly state otherwise,
      any Contribution intentionally submitted for inclusion in the Work
      by You to the Licensor shall be under the terms and conditions of
      this License, without any additional terms or conditions.
      Notwithstanding the above, nothing herein shall supersede or modify
      the terms of any separate license agreement you may have executed
      with Licensor regarding such Contributions.

   6. Trademarks. This License does not grant permission to use the trade
      names, trademarks, service marks, or product names of the Licensor,
      except as required for reasonable and customary use in describing the
      origin of the Work and reproducing the content of the NOTICE file.

   7. Disclaimer of Warranty. Unless required by applicable law or
      agreed to in writing, Licensor provides the Work (and each
      Contributor provides its Contributions) on an "AS IS" BASIS,
      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
      implied, including, without limitation, any warranties or conditions
      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A
      PARTICULAR PURPOSE. You are solely responsible for determining the
      appropriateness of using or redistributing the Work and assume any
      risks associated with Your exercise of permissions under this License.

   8. Limitation of Liability. In no event and under no legal theory,
      whether in tort (including negligence), contract, or otherwise,
      unless required by applicable law (such as deliberate and grossly
      negligent acts) or agreed to in writing, shall any Contributor be
      liable to You for damages, including any direct, indirect, special,
      incidental, or consequential damages of any character arising as a
      result of this License or out of the use or inability to use the
      Work (including but not limited to damages for loss of goodwill,
      work stoppage, computer failure or malfunction, or any and all
      other commercial damages or losses), even if such Contributor
      has been advised of the possibility of such damages.

   9. Accepting Warranty or Additional Liability. While redistributing
      the Work or Derivative Works thereof, You may choose to offer,
      and charge a fee for, acceptance of support, warranty, indemnity,
      or other liability obligations and/or rights consistent with this
      License. However, in accepting such obligations, You may act only
      on Your own behalf and on Your sole responsibility, not on behalf
      of any other Contributor, and only if You agree to indemnify,
      defend, and hold each Contributor harmless for any liability
      incurred by, or claims asserted against, such Contributor by reason
      of your accepting any such warranty or additional liability.

   END OF TERMS AND CONDITIONS

   APPENDIX: How to apply the Apache License to your work.

      To apply the Apache License to your work, attach the following
      boilerplate notice, with the fields enclosed by brackets "{}"
      replaced with your own identifying information. (Don't include
      the brackets!)  The text should be enclosed in the appropriate
      comment syntax for the file format. We also recommend that a
      file or class name and description of purpose be included on the
      same "printed page" as the copyright notice for easier
      identification within third-party archives.

   Copyright 2019 Ross Wightman

   Licensed under the Apache License, Version 2.0 (the "License");
   you may not use this file except in compliance with the License.
   You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

   Unless required by applicable law or agreed to in writing, software
   distributed under the License is distributed on an "AS IS" BASIS,
   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
   See the License for the specific language governing permissions and
   limitations under the License.

```
</details>


## Preserved historical source

Every original Version 33 cell remains below as an inactive raw cell with its original cell identifier. The exact original notebook bytes (203,753 bytes, SHA-256 `df02bfdc…`) are also preserved as a SHA-256-bound gzip/base64 snapshot in the notebook metadata, so the original JSON, outputs, cell types and metadata can be reconstructed exactly. The historical launcher and reader in those cells do not run in version 34.
